# Affective Computing - Programming Assignment 3

### Objective

Your task is to use the feature-level method to combine facial expression features and audio features. A multi-modal emotion recognition system is constructed to recognize happy versus sadness facial expressions (binary-class problem) by using a classifier training and testing structure.

The original data is based on lab1 and lab2, from ten actors acting happy and sadness behaviors. 
* Task 1: Subspace-based feature fusion method: In this case, z-score normalization is utilized. Please read “Fusing Gabor and LBP feature sets for kernel-based face recognition” and learn how to use subspace-based feature fusion method for multi-modal system.

* Task 2: Based on Task 1, use Canonical Correlation Analysis to calculate the correlation coefficients of facial expression and audio features. Finally, use CCA to build a multi-modal emotion recognition system. The method is described in one conference paper “Feature fusion method based on canonical correlation analysis and handwritten character recognition”
* Task 3: Based on Task 1, create a Leave-One-Subject-Out (LOSO) cross-validation to estimate the performance more reliably.

To produce emotion recognition case, Support Vector Machine (SVM) classifiers are trained.  50 videos from 5 participants are used to train the emotion recognition systems by using spatiotemporal features. The rest of the data (50 videos) are used to evaluate the performances of the trained recognition systems.

## Task 1. Subspace-based method  
Please read “Fusing Gabor and LBP feature sets for kernel-based face recognition” and apply their framework for the exercise. We use Support Vector Machine (SVM) with linear kernel for classification. As opposed to using Gabor features we are using the prosodic features from the last exercise.


### Setting up the environment 

First, we need to import the basic modules for loading the data and data processing

In [1]:
import sys
sys.path.append('../')
from skimage import io
from skimage import transform
from skimage import color
from skimage import img_as_ubyte
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import sklearn
import scipy.io as sio

c:\Users\mahya\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


### Loading data  <font color='red'>(0.5 point)</font>

We load the facial expression data (training data, training class, testing data, testing class) and audio data (training data, testing data)

In [2]:
mdata = sio.loadmat('lab3_data.mat')

#Facial expression training and testing data, training and testing class
training_data = mdata['training_data']
testing_data = mdata['testing_data']
training_class = mdata['training_class']
testing_class = mdata['testing_class']

#Audio training and testing data
training_data_proso = mdata['training_data_proso']
testing_data_proso = mdata['testing_data_proso']

### Extract the subspace for facial expression features and audio features <font color='red'>(2 point)</font>
Extract the subspace for facial expression features and audio features using principal component analysis through using **PCA class**.
The `reduced_dim` is the dimensionality of the reduced subspace.
Set `reduced_dim` to 20 and 15 for facial expression features and audio features, respectively. Normalization should be done subject wise. The test data should be normalized with the values from the training data.
For concatenating the features use the __[`np.concatenate()`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.concatenate.html)__ function.

You will implement the PCA class with two methods, **fit** and **transform**. The **fit** method takes one input array with no return values and the **transform** method takes one input array and returns a transformed array with dimensions. Use (__[`numpy.linalg.svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)__) for singular values extraction.

In [3]:
class PCA:
    """Principal component analysis (PCA).
    Parameters
    ----------
    n_components : int
        Number of principal components to use.
    whiten : bool, default=False
        When true, the output of transformed features is divided by the
        square root of the explained variance.
    Examples
    --------
    >>> import numpy as np
    >>> X = np.array([[-1, -1], [-2, -1], [-3, -2], [1, 1], [2, 1], [3, 2]])
    >>> pca = PCA(n_components=2)
    >>> pca.fit(X)
    >>> pca.transform(X)
    >>> array([[ 1.38340578,  0.2935787 ],
               [ 2.22189802, -0.25133484],
               [ 3.6053038 ,  0.04224385],
               [-1.38340578, -0.2935787 ],
               [-2.22189802,  0.25133484],
               [-3.6053038 , -0.04224385]])
    """
    def __init__(self, n_components: int, whiten: bool = False) -> None:
        self.n_components = n_components
        self.whiten = whiten
        self.selected_components = None
        self.mean = None 
                   
    def fit(self, X: np.ndarray) -> None:
        """Fit the model with X.
        Parameters
        ----------
        X : a numpy array with dimensions (n_samples, n_features)
        """        
        #Step 1: Find the mean, and center the data
        self.mean = np.mean(X)
        X = X - self.mean
        
        #Step2:  Find the Covariance
        cov = np.cov(X)

        #Step 3: Apply SVD and choose the components, make the hermitian argument True.
        U, S, Vt = np.linalg.svd(cov, hermitian=True)
        self.selected_components = Vt[:self.n_components]   # (n_components, n_features)
        # choose the singular values of diagnal matrix
        self.explained_variance = S[:self.n_components]
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Transform X with the fitted model.
        Parameters
        ----------
        X : a numpy array with dimensions (n_samples, n_features)
        
        Returns
        -------
        X_transformed: a numpy array with dimensions (n_samples, n_components)
        """
        # Center the data 
        X = X - np.mean(X)
        # Step 4: Choose and transform the features
        X_transformed = np.dot(X, self.selected_components.T)
        if self.whiten:
            # Normalize the transform features
            X_transformed /= np.sqrt(self.explained_variance)
        return X_transformed
        

In [4]:
from sklearn.decomposition import PCA 
from scipy import stats

#Set Reduced_dim for facial expression features and audio features, respectively.
reduced_dim_v = 20
reduced_dim_a = 15

#Extract the subspace for facial expression features though PCA. 
#If you are using sklearn use random_state=0, to ensure consistant results
pca_v = PCA(reduced_dim_v, random_state=0)
pca_v.fit(training_data)

#Transform training_data and testing data respectively
pca_v_training = pca_v.transform(training_data)
pca_v_testing = pca_v.transform(testing_data)

#Extract the subspace for audio features though PCA
pca_a = PCA(reduced_dim_a, random_state=0)
pca_a.fit(training_data_proso)

#Transform the training_data and testing_data respectively
pca_a_training = pca_a.transform(training_data_proso)
pca_a_testing = pca_a.transform(testing_data_proso)

#Normalize the features
# Idk why but using stats.zscore doesnt give the expected results
# pca_v_training = stats.zscore(pca_v_training, axis=0)
# pca_v_testing = stats.zscore(pca_v_testing, axis=0)

# pca_a_training = stats.zscore(pca_a_training, axis=0)
# pca_a_testing = stats.zscore(pca_a_testing, axis=0)

# ----------------------
pca_v_mean = np.mean(pca_v_training, axis=0)
pca_v_std = np.std(pca_v_training, axis=0)
pca_v_training = (pca_v_training - pca_v_mean) / pca_v_std
pca_v_testing = (pca_v_testing - pca_v_mean) / pca_v_std  

pca_a_mean = np.mean(pca_a_training, axis=0)
pca_a_std = np.std(pca_a_training, axis=0)
pca_a_training = (pca_a_training - pca_a_mean) / pca_a_std
pca_a_testing = (pca_a_testing - pca_a_mean) / pca_a_std  
# ----------------------

#Concatenate the transformed training data of facial expression features and audio features together
combined_train = np.concatenate((pca_v_training, pca_a_training), axis=1)

#Concatenate the transformed testing data of facial expression features and audio features together
combined_test = np.concatenate((pca_v_testing, pca_a_testing), axis=1)

### Question 1. Why is PCA used? Why not just concatenate the extracted features without PCA? <font color='red'>(0.5 point)</font>

### Your answer:

We actually can just concatenate the extracted features, but it is not efficient. Because the data might be noisy, and the feature space becomes large and high-dimensional. And it leads to an issue called curse of dimensionality which means that the model will struggle to find patterns because the data becomes too sparse.

Therefore, we are using PCA, which reduces the dimensionality of features while keeping the most important parts of them. It basically converts features to a more managable size. PCA helps the model to perform faster and better. An it can prevent overfitting.

### Feature classification <font color='red'>(0.5 point)</font>
Use the __[`SVM`](http://scikit-learn.org/stable/modules/svm.html)__ function to train Support Vector Machine (SVM) classifiers.
Construct a SVM using the combined training data and linear kernel. The `training_class` group vector contains the class of samples: 1 = happy, 2 = sadness, corresponding to the rows of the training data matrices.

Then, calculate average classification performances for both training and testing data. The correct class labels corresponding with the rows of the training and testing data matrices are in the variables ‘training_class’ and ‘testing_class’, respectively.

In [5]:
from sklearn import svm
from sklearn.metrics import accuracy_score

# Train SVM classifier
classifier = svm.SVC(kernel='linear', random_state=0)
classifier.fit(combined_train, training_class.ravel())

#The prediction results
train_preds = classifier.predict(combined_train)
test_preds = classifier.predict(combined_test)

#Calculate and print the training accuracy and testing accuracy. 
train_acc = accuracy_score(training_class, train_preds)
test_acc = accuracy_score(testing_class, test_preds)

print(train_acc)
print(test_acc)

1.0
0.98


### <font color='red'>(0.5 point)</font>
Compute the confusion matrices using __[`sklearn.metrics.confusion_matrix()`](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)__function for both the training data and testing data.


In [6]:
from sklearn.metrics import confusion_matrix

train_conf = confusion_matrix(training_class, train_preds)
test_conf = confusion_matrix(testing_class, test_preds)

print(train_conf)
print(test_conf)


[[25  0]
 [ 0 25]]
[[25  0]
 [ 1 24]]


## Task 2. 
As opposed to a simple concatenation we can try something smarter that utilizes the common characteristics of the fused features. This is achieved using the CCA. Use the PCA transformed vectors and set the number of components for the CCA to be 15.


### <font color='red'>(1 point)</font>

Use (__[`sklearn.cross_decomposition.CCA()`](http://scikit-learn.org/stable/modules/generated/sklearn.cross_decomposition.CCA.html)__) function to calculate the correlation coefficients of facial expression features and audio features. For `n_components` of CCA use the same number as the reduced dimensionality of the audio features in the previous task.

In [7]:
from sklearn.cross_decomposition import CCA
import numpy as np

#Use CCA to construct the Canonical Projective Vector (CPV)
cca = CCA(n_components=15)
cca.fit(pca_v_training, pca_a_training)

#Construct Canonical Correlation Discriminant Features (CCDF) for both the training data and testing data
ccdf_train_v, ccdf_train_a = cca.transform(pca_v_training, pca_a_training)
ccdf_test_v, ccdf_test_a = cca.transform(pca_v_testing, pca_a_testing)

# Concatenate the CCA transformed features for training data and testing data
combined_train_ccdf = np.concatenate((ccdf_train_v, ccdf_train_a), axis=1)
combined_test_ccdf = np.concatenate((ccdf_test_v, ccdf_test_a), axis=1)


### <font color='red'>(1 point)</font>
Train a SVM classifier using a linear kernel, print the training and testing accuracy and compute the confusion matrix.

In [11]:
#Train svm classifier 
classifier = svm.SVC(kernel='linear')
classifier.fit(combined_train_ccdf, training_class.ravel())

#The prediction results
train_preds = classifier.predict(combined_train_ccdf)
test_preds = classifier.predict(combined_test_ccdf)

#Calculate and print the training accuracy and testing accuracy. 
train_acc = accuracy_score(training_class, train_preds)
test_acc = accuracy_score(testing_class, test_preds)

print(train_acc)
print(test_acc)

# Compute the confusion matrix using sklearn.metrics.confusion_matrix() function for training data and testing data respectively
train_conf = confusion_matrix(training_class, train_preds)
test_conf = confusion_matrix(testing_class, test_preds)

print(train_conf)
print(test_conf)

1.0
0.92
[[25  0]
 [ 0 25]]
[[25  0]
 [ 4 21]]


### Question 2. In this exercise a feature-level method was used to fuse the features. What are the other types of methods for data fusion? <font color='red'>(0.5 point)</font>

### Your answer:

Data fusion can be done in different levels. here are some examples and methods of data fusion:

Decision-level fusion: in this method, each data source is processed separately, and the results are integrated with each other to make a final decision. This fusion is often used in classification tasks where the outputs of multiple models are combined.

Sensor Fusion: This specific type of data fusion combines data from multiple sensors to improve the accuracy and reliability of the collected dataset.

Data-level Fusion: This method combines raw data from multiple sources before any feature extraction. It allows for a holistic view of the data.

When I searched for this topic, I found other levels of data fusion like temporal, hierarchical, decision level, etc too.

### Question 3. Compare the results from all the the different methods from assignments 1, 2 and 3. What method performed the best? What was the worst? Hypothesize as to why certain methods performed better than others. <font color='red'>(0.5 point)</font>

### Your answer:

So these are the results from all three exercises:

Assignment 1: Facial Expression Analysis
- Training accuracy: 0.88
- Testing accuracy: 0.72

Assignment 2: Speech Recognition for Emotion
- Training accuracy (Prosodic): 0.84
- Testing accuracy (Prosodic): 0.62
- Training accuracy (MFCC): 0.96
- Testing accuracy (MFCC): 0.84

Assignment 3: Multi-Modal Expression Analysis
- Training accuracy: 1
- Testing accuracy: 0.92

As you can see, multi-modal approach has the best performance, while the prosodic features performed the worst. We can conclude that the multi-modal expression analysis method worked as the most effective approach, since it benefited from the strengths of both audio and visual data. In contrast, the speech recognition method using prosodic features alone had the weakest performance, which shows that this feature set may not capture the emotional cues as well as others. Prosodic features capture aspects like pitch and intonation but may miss important points in data.

## Task 3: 
For a more reliable evaluation, often the Leave-One-Subject-Out (LOSO) cross-validation is used instead of the common train-test split. Cross-validation gives us a more reliable measure of the performance as all of the data is used for both training and testing. LOSO is used as emotions are highly dependent on the subject. By using LOSO, we guarantee that a subject is always in either the training or testing data and not in both.

* Join the training/testing data matrices and the class vectors. Combine also the ‘training_data_personID’ and ‘testing_data_personID’ vectors.

* Assume we have a total of $n$ subjects. Now, we will create a total of $n$ folds (loops), where each folds' training set contains the data from $n-1$ subjects and the testing set consists of only $1$ subject.

* Follow the steps taken in the first task: project the data to a subspace using PCA, conatenate the audio and video features together, train an SVM and finally evaluate the performance.

* The solution should be able to generalize over different numbers of subjects and samples, *e.g.*, a dataset may have 24 subjects, where subject1 has 4 samples and subject2 has 32 samples.

### <font color='red'>(0.5 point)</font>

In [12]:
mdata = sio.loadmat('lab3_data.mat')

#Combine the training data, testing data,label and persion ID for video and audio respectively, in order to get the whole dataset. 
lbp_data = np.concatenate((mdata['training_data'], mdata['testing_data']), axis=0)
proso_data = np.concatenate((mdata['training_data_proso'], mdata['testing_data_proso']), axis=0)

labels = np.concatenate((mdata['training_class'].flatten(), mdata['testing_class'].flatten()), axis=0)
subjects = np.concatenate((mdata['training_personID'].flatten(), mdata['testing_personID'].flatten()), axis=0)

#Get the number of the subject
subject_ids = np.unique(subjects)

#Print the shapes and the list of subject_ids for a sanity check
print(f'Shape of lbp_data: {lbp_data.shape}')
print(f'Shape of proso_data: {proso_data.shape}')
print(f'Shape of labels: {labels.shape}')
print(f'Shape of subjects: {subjects.shape}')
print(f'Value of subject_ids: {subject_ids}')

Shape of lbp_data: (100, 708)
Shape of proso_data: (100, 15)
Shape of labels: (100,)
Shape of subjects: (100,)
Value of subject_ids: [ 1  2  3  4  5  7  8  9 10 12]


### <font color='red'>(2 point)</font>

In [13]:
accuracies = []
#Loop over each subject
for subject_id in subject_ids:
    #Create a boolean array for the training and testing set indices
    #The train_idx should be a list of form [True, True, False, ...], where True indicates the position
    #for the samples that are not the current subject_id
    train_idx = np.array([True if x != subject_id else False for x in subjects])
    #Similar for the test_idx, True indicates the position of the current subject_id
    test_idx = np.array([False if x != subject_id else True for x in subjects])
    
    #Create the training and testing sets for lbp, proso and labels by indexing lbp_data, proso_data and labels
    #with the boolean arrays train_idx and test_idx
    lbp_train = lbp_data[train_idx]
    lbp_test = lbp_data[test_idx]
    proso_train = proso_data[train_idx]
    proso_test = proso_data[test_idx]
    labels_train = labels[train_idx]
    labels_test = labels[test_idx]
    
    #Create the PCA for both lbp and proso. We take a slight shortcut compared to task 1,
    #by using the whiten=True parameter for normalizing the features. This means that
    #there is no need for normalization afterwards
    pca_v = PCA(n_components=20, whiten=True)
    pca_a = PCA(n_components=15, whiten=True)
    
    #Fit the PCAs with the training data
    pca_v.fit(lbp_train)
    pca_a.fit(proso_train)
    
    #Transform both the training and testing data with the PCA
    pca_v_train = pca_v.transform(lbp_train)
    pca_v_test = pca_v.transform(lbp_test)

    pca_a_train = pca_a.transform(proso_train)
    pca_a_test = pca_a.transform(proso_test)

    #Concatenate the features together
    combined_train = np.concatenate((pca_v_train, pca_a_train), axis = 1)
    combined_test = np.concatenate((pca_v_test, pca_a_test), axis = 1)
    
    #Create a linear SVM and train it
    classifier = svm.SVC(kernel='linear', random_state=0)
    classifier.fit(combined_train, labels_train)
    
    #Calculate the accuracy for the testing data and add it to the list of accuracies
    test_preds = classifier.predict(combined_test)
    test_acc = accuracy_score(labels_test, test_preds)

    accuracies.append(test_acc)
    
#Calculate the average of the accuracies. Print both the list of accuracies and the average    
print(f'accuracies: {accuracies}')
print(f'mean of accuracy: {np.mean(accuracies)}')

accuracies: [0.9, 0.8, 1.0, 0.9, 0.9, 1.0, 1.0, 1.0, 0.8, 1.0]
mean of accuracy: 0.93


### Question 4. The accuracy of LOSO (0.93) is lower than the accuracy achieved by the train-test split (0.98) in task 1. Hypothesize as to why the two are different. Which one is better for evaluation?  <font color='red'>(0.25 point)</font>

### Your answer:

In LOSO, one subject is left out for testing, so the model is evaluated on data it has never seen before. This can lead to lower accuracy because the model might not generalize well to unseen subjects. In contrast, the train-test split can lead to higher accuracy since it may include data from the same subjects in both training and testing, allowing the model to find and extract similarities, resulting in better accuracy. Also, it may even lead to overfitting in a certain subject. 

LOSO is generally better for evaluation. Because in real-life senarios subject variability is really important. Loso provides a more realistic measure of how well the model will perform on unseen data.

### Question 5. In PCA why `whiten` parametere is better and why it replaces the normalization?  <font color='red'>(0.25 point)</font>

### Your answer:

The whiten parameter scales the transformed data so that each principal component has unit variance. In this way, the model won't have any bias toward the features having a high variance. And all features corporate equally. Whitening also eliminates the need for additional z-score normalization because it scales each component by the inverse of its standard deviation.